In [37]:
import polars as pl
import os

from pathlib import Path

In [38]:
DONOR = 'donor_3'

In [39]:
WORKING_PATH = Path('/group/pmc021/amunif/epi-thesis/workflow/16_Pairwise Ranking Healthy Liver/')
DATASET_PATH = WORKING_PATH / 'dataset'/ DONOR
OUTPUT_PATH  = WORKING_PATH / 'output' / 'combined' / DONOR

# Merge the experiment results

In [40]:
# Read all CSV into single dataframe
pl_df = pl.read_csv(OUTPUT_PATH / 'test' / "*-test-metrics.csv")

In [41]:
pl_df

model,seed,histone_marker,epochs_trained,val_accuracy,val_auc,test_accuracy,test_auc,test_aucprc,test_precision,test_recall,test_f1,antisymmetry
str,i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""DirectRanker""",1011,"""H3K4me3""",84,71.4,0.79,71.9,0.8161,0.8074,0.7035,0.761,0.7311,0.894
"""LogisticRegression""",1011,"""H3K4me3""",null,70.0,0.7628,71.0,0.7955,0.7867,0.75,0.6335,0.6868,0.793
"""RandomForest""",1011,"""H3K4me3""",null,70.2,0.7707,73.1,0.8097,0.7992,0.7716,0.6594,0.7111,0.775
"""SVM_Linear""",1011,"""H3K4me3""",null,70.1,0.7646,71.3,0.7959,0.7882,0.7541,0.6355,0.6897,0.794
"""DirectRanker""",123,"""H3K4me3""",60,74.8,0.84,71.7,0.8055,0.7782,0.6917,0.7663,0.7271,0.899
…,…,…,…,…,…,…,…,…,…,…,…,…
"""SVM_Linear""",456,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",null,72.1,0.7833,70.9,0.7483,0.7158,0.7231,0.6502,0.6847,0.757
"""DirectRanker""",789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",67,73.4,0.81,74.6,0.8397,0.8304,0.7371,0.7694,0.7529,0.96
"""LogisticRegression""",789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",null,71.9,0.7735,71.6,0.781,0.7574,0.7483,0.6561,0.6992,0.761


In [42]:
# Save the results by model 
MODEL_RESULT_FOLDER = OUTPUT_PATH / 'model_results'
os.makedirs(MODEL_RESULT_FOLDER, exist_ok=True)

for model_name in pl_df["model"].unique():
    pl_df.filter(pl_df["model"] == model_name).write_csv(MODEL_RESULT_FOLDER / f"{model_name}.csv", include_header=True)

In [43]:
# Save the results for all models and all seeds
pl_df.write_csv(OUTPUT_PATH / f"{DONOR}_all_results.csv", include_header=True)

In [52]:
summary_df = (
    pl_df
    .group_by(["histone_marker", "model"])
    .agg([
        pl.col("val_accuracy").mean().alias("val_accuracy_mean"),
        pl.col("val_accuracy").std().alias("val_accuracy_std"),
        pl.col("val_auc").mean().alias("val_auc_mean"),
        pl.col("val_auc").std().alias("val_auc_std"),
        pl.col("test_accuracy").mean().alias("test_accuracy_mean"),
        pl.col("test_accuracy").std().alias("test_accuracy_std"),
        pl.col("test_auc").mean().alias("test_auc_mean"),
        pl.col("test_auc").std().alias("test_auc_std"),
        pl.col("test_aucprc").mean().alias("test_aucprc_mean"),
        pl.col("test_aucprc").std().alias("test_aucprc_std"),
        pl.col("antisymmetry").mean().alias("antisymmetry_mean"),
        pl.col("antisymmetry").std().alias("antisymmetry_std")
    ])
    .sort(["histone_marker", "test_accuracy_mean"], descending=[False, True])
    .with_columns(pl.col(pl.Float64).round(4))
)

In [53]:
summary_df

histone_marker,model,val_accuracy_mean,val_accuracy_std,val_auc_mean,val_auc_std,test_accuracy_mean,test_accuracy_std,test_auc_mean,test_auc_std,test_aucprc_mean,test_aucprc_std,antisymmetry_mean,antisymmetry_std
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""H3K27ac""","""RandomForest""",74.46,2.5696,0.8276,0.0251,75.08,1.4202,0.8378,0.0117,0.826,0.0175,0.7988,0.01
"""H3K27ac""","""LogisticRegression""",73.14,2.4089,0.7942,0.022,74.3,1.4422,0.8086,0.0153,0.7926,0.0157,0.7904,0.0205
"""H3K27ac""","""SVM_Linear""",73.16,2.1197,0.7942,0.0212,73.94,1.5453,0.8097,0.0155,0.7932,0.0153,0.7872,0.0191
"""H3K27ac""","""DirectRanker""",73.28,1.7655,0.82,0.0245,73.84,1.0945,0.8281,0.0147,0.8137,0.0163,0.8944,0.0054
"""H3K27ac-H3K27me3""","""RandomForest""",74.32,2.9828,0.8295,0.0259,75.1,1.6748,0.8367,0.0139,0.8279,0.0193,0.8188,0.0083
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""H3K9me3-H3K27ac-H3K27me3""","""SVM_Linear""",70.26,2.6444,0.7586,0.0256,71.1,1.9962,0.7694,0.0182,0.7439,0.0223,0.7806,0.0191
"""H3K9me3-H3K27me3""","""DirectRanker""",56.76,1.7672,0.63,0.0141,56.34,1.1845,0.6181,0.0153,0.5805,0.0134,0.2964,0.0176
"""H3K9me3-H3K27me3""","""RandomForest""",57.42,2.4763,0.6237,0.0227,56.22,0.9391,0.6058,0.0128,0.5722,0.0117,0.2668,0.0223


In [54]:
summary_df.write_csv(OUTPUT_PATH/ f"{DONOR}_result_summary.csv", include_header=True)

In [55]:
for model_name in summary_df["model"].unique():
    summary_df.filter(summary_df["model"] == model_name).write_csv(MODEL_RESULT_FOLDER / f"{model_name}_summary.csv", include_header=True)

# Merge the label distribution

In [56]:
# Read all label distribution CSV into single dataframe
label_df = pl.read_csv(OUTPUT_PATH / 'test' / "*-label-distribution.csv")

In [57]:
label_df

seed,histone_marker,split,label,count,total,percentage
i64,str,str,i64,i64,i64,f64
1011,"""H3K4me3""","""Train""",0,4082,8000,51.02
1011,"""H3K4me3""","""Train""",1,3918,8000,48.98
1011,"""H3K4me3""","""Val""",0,518,1000,51.8
1011,"""H3K4me3""","""Val""",1,482,1000,48.2
1011,"""H3K4me3""","""Test""",0,498,1000,49.8
…,…,…,…,…,…,…
789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…","""Train""",1,3923,8000,49.04
789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…","""Val""",0,529,1000,52.9
789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…","""Val""",1,471,1000,47.1


In [58]:
# Make the summary by seed, split and label
label_summary_df = (
    label_df
    .group_by(['split', 'label'])
    .agg(pl.col('percentage').mean().round(2).alias('average_percentage_%'))
    .sort(['split', 'label'])
)

In [59]:
label_summary_df

split,label,average_percentage_%
str,i64,f64
"""Test""",0,50.72
"""Test""",1,49.28
"""Train""",0,50.99
"""Train""",1,49.01
"""Val""",0,52.04
"""Val""",1,47.96
